In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_cerebras import ChatCerebras

llm = ChatCerebras(model="llama3.1-8b", temperature=0.7, max_tokens=800)
ai_msg = llm.invoke("what is mean by cerebras")

ai_msg.content

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
load_dotenv()

llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

In [3]:
result = llm.invoke("Plan a three-step research workflow for competitive analysis.")
print(result.content)

Here's a concise, actionable **three-step research workflow for competitive analysis** designed to drive strategic decisions—not just collect data. Each step builds on the last, ensuring focus, depth, and practical output:

---

### **Step 1: Define Scope & Identify Key Competitors (The "Who" and "Why")**  
*Goal: Avoid analysis paralysis by focusing only on relevant players tied to your strategic objectives.*  
- **Clarify Your Objective**: Start with a specific business question (e.g., *"How do we gain share in the mid-tier SaaS market by Q3?"* or *"What pricing tactics are eroding our margins in Enterprise?"*).  
- **Tier Competitors Strategically**:  
  - **Direct Competitors**: Same product/service, same target audience (e.g., Salesforce vs. HubSpot for CRM).  
  - **Indirect/Adjacent Competitors**: Solve the same customer problem differently (e.g., Zoom vs. Slack for team communication).  
  - **Emerging/Disruptive Threats**: Startups, tech shifts, or regulatory changes (e.g., AI

In [4]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from dotenv import load_dotenv
load_dotenv()

# embeddings = NVIDIAEmbeddings(model="nvidia/nv-embed-v1")
embeddings = NVIDIAEmbeddings(model="baai/bge-m3")

# Single text
vector = embeddings.embed_query("What is competitive analysis?")
print(len(vector))  # dimension size

# Multiple texts
vectors = embeddings.embed_documents([
    "Competitive analysis helps identify market position.",
    "SWOT analysis is a common framework.",
    "Porter's five forces is another approach."
])
print(len(vectors))       # number of docs
print(len(vectors[0]))    # dimension of first vector

1024
3
1024


In [5]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embedder = NVIDIAEmbeddings(model="baai/bge-m3")
embeddings = embedder.embed_query("What's the temperature today?")

print(len(embeddings))

1024


In [6]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from dotenv import load_dotenv
load_dotenv()

embeddings = NVIDIAEmbeddings(model="baai/bge-m3")

vector = embeddings.embed_query("What is competitive analysis?")

print(f"Vector dimensions : {len(vector)}")
print(f"First 10 values   : {vector[:10]}")
print(f"Last  10 values   : {vector[-10:]}")
print(f"Min value         : {min(vector):.6f}")
print(f"Max value         : {max(vector):.6f}")

# multiple docs
texts = [
    "Competitive analysis helps identify market position.",
    "SWOT analysis is a common framework.",
    "Porter's five forces is another approach.",
]
vectors = embeddings.embed_documents(texts)

print(f"\nTotal docs embedded : {len(vectors)}")
print(f"Each vector size    : {len(vectors[0])}")
for i, (text, vec) in enumerate(zip(texts, vectors)):
    print(f"\n  doc #{i+1} : '{text[:50]}'")
    print(f"  first 5 values : {vec[:5]}")

Vector dimensions : 1024
First 10 values   : [-0.10760498046875, 0.003124237060546875, -0.0230712890625, 0.0156097412109375, 0.0286712646484375, 0.0281524658203125, -0.006671905517578125, 0.033935546875, 0.0129241943359375, 0.019683837890625]
Last  10 values   : [0.02471923828125, 0.0186767578125, 0.048736572265625, 0.0093231201171875, -0.033721923828125, -0.048126220703125, 0.006641387939453125, -0.03839111328125, -0.00080108642578125, -0.0079803466796875]
Min value         : -0.154785
Max value         : 0.220459

Total docs embedded : 3
Each vector size    : 1024

  doc #1 : 'Competitive analysis helps identify market positio'
  first 5 values : [-0.08746337890625, -0.007678985595703125, -0.0438232421875, -0.006526947021484375, -0.007617950439453125]

  doc #2 : 'SWOT analysis is a common framework.'
  first 5 values : [-0.061004638671875, 0.0165863037109375, -0.076416015625, -0.017822265625, 0.001003265380859375]

  doc #3 : 'Porter's five forces is another approach.'
  first 5 val

In [7]:
import os
import json
from unstructured_pytesseract import pytesseract

pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata"  # optional

from collections import defaultdict
from unstructured.partition.pdf import partition_pdf


os.makedirs("./extracted_images", exist_ok=True)

# extract every possible content type in one call
elements = partition_pdf(
    filename="attention.pdf",
    strategy="hi_res",                              # required for layout, tables, images
    infer_table_structure=True,                     # structured HTML for tables
    extract_image_block_types=["Image", "Table"],   # save image blocks to disk
    extract_image_block_output_dir="./extracted_images",
    extract_image_block_to_payload=False,
    include_page_breaks=True,                       # capture PageBreak elements
    languages=["eng"],
    detect_language_per_element=True,               # per-element language detection
    hi_res_model_name="yolox",
    unique_element_ids=True,                        # UUID per element instead of hash
    starting_page_number=1,
)

print(f"Total elements extracted: {len(elements)}")

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Total elements extracted: 247


In [8]:
grouped = defaultdict(list)
for el in elements:
    grouped[type(el).__name__].append(el)

print("\nType distribution:")
for type_name, els in sorted(grouped.items()):
    print(f"  {type_name:25s} : {len(els)}")


Type distribution:
  FigureCaption             : 5
  Footer                    : 7
  Formula                   : 4
  Header                    : 3
  Image                     : 7
  ListItem                  : 43
  NarrativeText             : 81
  PageBreak                 : 15
  Table                     : 4
  Text                      : 52
  Title                     : 26


In [9]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(elements, max_characters=1000, overlap=50)

langchain_docs = [
    Document(
        page_content=chunk.text,
        metadata={
            "source"  : chunk.metadata.filename,
            "page"  : chunk.metadata.page_number,
            "type" : type(chunk).__name__,
        }
    )
    for chunk in chunks
    if chunk.text.strip()
]


In [10]:
# build FAISS index
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"\nFAISS index built — total vectors: {vectorstore.index.ntotal}")


FAISS index built — total vectors: 54


In [11]:
query = "What is self attention"

# plain similarity search
results = vectorstore.similarity_search(query, k=3)

In [12]:
print(f"\nTop {len(results)} results for: '{query}'")
for i, doc in enumerate(results):
    print(f"\n  Result #{i+1}")
    print(f"  content  : {doc.page_content[:200]}")
    print(f"  metadata : {doc.metadata}")


Top 3 results for: 'What is self attention'

  Result #1
  content  : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has be
  metadata : {'source': 'attention.pdf', 'page': 2, 'type': 'CompositeElement'}

  Result #2
  content  : • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward informatio
  metadata : {'source': 'attention.pdf', 'page': 5, 'type': 'CompositeElement'}

  Result #3
  content  : 3.2 Attention

An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as
  metadata : {'source': 'attention.pdf', 'page': 3, 'type': 'CompositeElement'}


In [13]:
# similarity search with relevance scores
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

In [15]:
print(f"\nResults with scores:")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n  Result #{i+1} | score={score:.6f}")
    print(f"  content  : {doc.page_content[:150]}")


Results with scores:

  Result #1 | score=0.730669
  content  : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a rep

  Result #2 | score=0.950693
  content  : • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including tha

  Result #3 | score=0.976797
  content  : 3.2 Attention

An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and


In [16]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)


In [17]:

retrieved_docs = retriever.invoke("What is the encoder?")

In [18]:
print(f"\nRetrieved {len(retrieved_docs)} docs")
for i, doc in enumerate(retrieved_docs):
    print(f"\n  doc #{i+1} | page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
    print(f"  {doc.page_content[:200]}")


Retrieved 4 docs

  doc #1 | page=3 | source=attention.pdf
  The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, 

  doc #2 | page=3 | source=attention.pdf
  Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head at

  doc #3 | page=2 | source=attention.pdf
  3 Model Architecture

Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) 

  doc #4 | page=6 | source=attention.pdf
  Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (r


In [19]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
""")

def format_docs(docs: list) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


In [20]:
qa_chain = ({
        "context":  retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
# sample questions
questions = [
    "What is mean by the transformer architecture?",
    "explain the positional encoding",
    "What is Multi-Head Attention",
]

for q in questions:
    print(f"\nQ: {q}")
    answer = qa_chain.invoke(q)
    print(f"A: {answer}")


Q: What is mean by the transformer architecture?
A: The Transformer architecture is a model that uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, instead of relying on recurrent networks. It is the first sequence transduction model based entirely on attention, replacing the recurrent layers commonly used in encoder-decoder architectures with multi-headed self-attention. This allows for significantly more parallelization and can reach a new state of the art in translation quality.

Q: explain the positional encoding
A: The positional encoding is a mechanism used to inject information about the relative or absolute position of tokens in a sequence into the input embeddings. This is necessary for the model to make use of the order of the sequence, since the model contains no recurrence and no convolution.

In this work, the positional encoding is implemented using sine and cosine functions of different frequencies. The encoding is calcu

In [22]:
questions = [
    "What is mean by the transformer architecture?",
    "explain the positional encoding",
    "What is Multi-Head Attention",
]

for q in questions:
    print(f"\n\n\nQUERY: {q}")

    results = vectorstore.similarity_search_with_score(q, k=3)

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\n  Rank #{rank}")
        print(f"  score       : {score:.6f}")
        print(f"  source      : {doc.metadata.get('source')}")
        print(f"  page        : {doc.metadata.get('page')}")
        print(f"  type        : {doc.metadata.get('type')}")
        print(f"  content     :\n    {doc.page_content}")

print(f"  {'\n'}")




QUERY: What is mean by the transformer architecture?

  Rank #1
  score       : 0.902906
  source      : attention.pdf
  page        : 3
  type        : CompositeElement
  content     :
    The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.

3.1 Encoder and Decoder Stacks

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as we

In [23]:
from unstructured.staging.base import elements_to_json

elements_to_json(elements, filename="attention_all_elements.json", indent=2)
print("\nAll elements saved to attention_all_elements.json")

# verify by reloading
from unstructured.staging.base import elements_from_json
reloaded = elements_from_json(filename="attention_all_elements.json")
print(f"Reloaded {len(reloaded)} elements from JSON")


All elements saved to attention_all_elements.json
Reloaded 247 elements from JSON


In [24]:
import json

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# top-level keys
top_keys = set()
for record in data:
    top_keys.update(record.keys())

print(f"Top-level keys: {sorted(top_keys)}")

# metadata keys
meta_keys = set()
for record in data:
    meta = record.get("metadata", {})
    meta_keys.update(meta.keys())

print(f"\nMetadata keys: {sorted(meta_keys)}")

Top-level keys: ['element_id', 'metadata', 'text', 'type']

Metadata keys: ['coordinates', 'detection_class_prob', 'filename', 'filetype', 'image_path', 'is_extracted', 'languages', 'last_modified', 'page_number', 'parent_id', 'text_as_html']


In [25]:
import json
import random

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

samples = random.sample(data, 5)

for i, record in enumerate(samples, start=1):
    print(f"Sample #{i}")
    print(f"  type    : {record['type']}")
    print(f"  text    : {record['text']}")

Sample #1
  type    : ListItem
  text    : • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9].
Sample #2
  type    : NarrativeText
  text    : For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In the former task our best model outperforms even all previously reported ensembles.
Sample #3
  type    : FigureCaption
  text    : Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 t

In [26]:
import json

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

images = [record for record in data if record["type"] == "Image"]

print(f"Total images: {len(images)}")

for i, record in enumerate(images, start=1):
    print(f"Image #{i}")
    print(f"  page       : {record['metadata'].get('page_number')}")
    print(f"  image_path : {record['metadata'].get('image_path')}")
    print(f"  text       : {record['text']}")

Total images: 7
Image #1
  page       : 3
  image_path : ./extracted_images\figure-3-1.jpg
  text       : Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding
Image #2
  page       : 4
  image_path : ./extracted_images\figure-4-2.jpg
  text       : 
Image #3
  page       : 4
  image_path : ./extracted_images\figure-4-3.jpg
  text       : EEE Scaled Dot-Product | Attention 4
Image #4
  page       : 13
  image_path : ./extracted_images\figure-13-4.jpg
  text       : n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegse8Be, 42 PS8F SERRERR 268 PT, FESGaeavoezreosHePi_seecet wogcaadnad ~¥ 2®E& SoS vo FEotToec ace RHDNESLSOSGETBD.VVVV VV Vv YF neone er oOyS>H CHYVOD 9 X2Q OD < "A AAAAA A “=e 5 f SFogezsoogs S$ 8S

In [27]:
import json

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

images = [r for r in data if r["type"] == "Image"]

print(f"Total images: {len(images)}")


for i, img in enumerate(images, start=1):
    text = img["text"] or ""
    word_count = len(text.split())
    is_garbage = word_count < 5 or any(c in text for c in ["£", "¥", "=|", "ae,"])

    print(f"\nImage #{i} | page={img['metadata'].get('page_number')}")
    print(f"  text length : {len(text)} chars | words: {word_count}")
    print(f"  usable      : {'NO — garbled OCR' if is_garbage else 'YES'}")
    print(f"  text preview: {text[:100]}")

Total images: 7

Image #1 | page=3
  text length : 319 chars | words: 54
  usable      : NO — garbled OCR
  text preview: Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & 

Image #2 | page=4
  text length : 0 chars | words: 0
  usable      : NO — garbled OCR
  text preview: 

Image #3 | page=4
  text length : 36 chars | words: 6
  usable      : YES
  text preview: EEE Scaled Dot-Product | Attention 4

Image #4 | page=13
  text length : 384 chars | words: 89
  usable      : NO — garbled OCR
  text preview: n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegse8Be, 42 PS8F SERRERR 268 PT, FE

Image #5 | page=14
  text length : 458 chars | words: 96
  usable      : YES
  text preview: <ped> <ped> UOIUIGO == = uoluIdo Aw — Aw ul ul Bulssiw Bulssiw ae » ale aM: aM JEUM « yeEUM sl sl SI

Image #6 | page=15
  text length : 252 chars | words: 55
  usable      : YES
  text preview: <ped> <ped> <SOd>\ <SOd> UOIUICO =. uoluido Aw d

In [28]:
import json
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()

embeddings = NVIDIAEmbeddings(model="baai/bge-m3")

# load all elements
with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# build LangChain docs — include type in metadata so we can track it
langchain_docs = [
    Document(
        page_content=record["text"],
        metadata={
            "type"       : record["type"],
            "page"       : record["metadata"].get("page_number"),
            "element_id" : record["element_id"],
            "image_path" : record["metadata"].get("image_path"),   # None for non-images
        }
    )
    for record in data
    if record["text"] and record["text"].strip()   # skip empty text elements
]

print(f"Total docs going into FAISS: {len(langchain_docs)}")

# how many are images
image_docs = [d for d in langchain_docs if d.metadata["type"] == "Image"]
print(f"Image docs included        : {len(image_docs)}")

# build index
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"FAISS index size           : {vectorstore.index.ntotal}")

Total docs going into FAISS: 231
Image docs included        : 6
FAISS index size           : 231


In [29]:
import json
import re

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

images = [r for r in data if r["type"] == "Image"]

GARBAGE_PATTERNS = [
    r"<ped>", r"<SOd>", r"<SOA>",
    r"[£¥§©®°±²³¼½¾]",
    r"\b[A-Z]{6,}\b",
    r"={3,}",
    r"\w{1}\s\w{1}\s\w{1}\s\w{1}",
]

def is_garbage(text: str) -> bool:
    if not text or len(text.strip()) < 10:
        return True
    for pattern in GARBAGE_PATTERNS:
        if re.search(pattern, text):
            return True
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / len(text) > 0.20:
        return True
    return False

print(f"Total images: {len(images)}")


for i, img in enumerate(images, start=1):
    text       = img["text"] or ""
    image_path = img["metadata"].get("image_path") or "not saved"
    garbage    = is_garbage(text)
    status     = "GARBAGE" if garbage else "USABLE"

    print(f"\nImage #{i} | page={img['metadata'].get('page_number')} | {status}")
    print(f"  disk path : {image_path}")
    print(f"  chars={len(text)} | words={len(text.split())}")
    print(f"  preview   : {text[:120] if text else 'empty'}")


Total images: 7

Image #1 | page=3 | USABLE
  disk path : ./extracted_images\figure-3-1.jpg
  chars=319 | words=54
  preview   : Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx

Image #2 | page=4 | GARBAGE
  disk path : ./extracted_images\figure-4-2.jpg
  chars=0 | words=0
  preview   : empty

Image #3 | page=4 | USABLE
  disk path : ./extracted_images\figure-4-3.jpg
  chars=36 | words=6
  preview   : EEE Scaled Dot-Product | Attention 4

Image #4 | page=13 | GARBAGE
  disk path : ./extracted_images\figure-13-4.jpg
  chars=384 | words=89
  preview   : n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegse8Be, 42 PS8F SERRERR 268 PT, FESGaeavoezreosHePi_se

Image #5 | page=14 | GARBAGE
  disk path : ./extracted_images\figure-14-5.jpg
  chars=458 | words=96
  preview   : <ped> <ped> UOIUIGO == = uoluIdo Aw — Aw ul ul Bulssiw Bulssiw ae » ale aM: aM JEUM « yeEUM sl sl SIU] SIU} ysn/ isn aq 

Image #6 |

In [32]:
from langchain_core.documents import Document

langchain_docs = []

for record in data:
    text = record["text"] or ""

    # skip completely empty
    if not text.strip():
        continue

    # skip garbage image OCR
    if record["type"] == "Image" and is_garbage(text):
        print(f"Skipped garbage image: page={record['metadata'].get('page_number')}")
        continue

    langchain_docs.append(Document(
        page_content=text,
        metadata={
            "type"       : record["type"],
            "page"       : record["metadata"].get("page_number"),
            "element_id" : record["element_id"],
            "image_path" : record["metadata"].get("image_path"),
        }
    ))

# breakdown of what went in
from collections import Counter
type_counts = Counter(d.metadata["type"] for d in langchain_docs)
print(f"\nDocs in FAISS: {len(langchain_docs)}")
for t, c in sorted(type_counts.items()):
    print(f"  {t:20s} : {c}")

vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"\nFAISS index size: {vectorstore.index.ntotal}")

Skipped garbage image: page=13
Skipped garbage image: page=14
Skipped garbage image: page=15
Skipped garbage image: page=15

Docs in FAISS: 227
  FigureCaption        : 5
  Footer               : 7
  Formula              : 4
  Header               : 3
  Image                : 2
  ListItem             : 43
  NarrativeText        : 81
  Table                : 4
  Title                : 26
  UncategorizedText    : 52

FAISS index size: 227


In [ ]:
queries = [
    "What is the transformer architecture?",
    "explain the positional encoding",
    "What is Multi-Head Attention",
]

for q in queries:
    print(f"QUERY: {q}")

    results = vectorstore.similarity_search_with_score(q, k=4)

    type_hits = Counter(doc.metadata["type"] for doc, _ in results)
    print(f"Type hits in top 4: {dict(type_hits)}")

    for rank, (doc, score) in enumerate(results, start=1):
        elem_type = doc.metadata.get("type")
        print(f"\n  Rank #{rank} | score={score:.4f} | type={elem_type} | page={doc.metadata.get('page')}")
        print(f"  content : {doc.page_content[:200]}")


QUERY: What is the transformer architecture?
Type hits in top 4: {'FigureCaption': 1, 'NarrativeText': 3}

  Rank #1 | score=0.8350 | type=FigureCaption | page=3
  content : Figure 1: The Transformer - model architecture.

  Rank #2 | score=0.8679 | type=NarrativeText | page=2
  content : In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Tran

  Rank #3 | score=0.8992 | type=NarrativeText | page=3
  content : The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, 

  Rank #4 | score=0.9236 | type=NarrativeText | page=1
  content : The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models

In [35]:
import json
import re
from collections import Counter
from langchain_core.documents import Document

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

GARBAGE_PATTERNS = [
    r"<ped>", r"<SOd>", r"<SOA>",
    r"[£¥§©®°±²³¼½¾]",
    r"\b[A-Z]{6,}\b",
    r"={3,}",
    r"\w{1}\s\w{1}\s\w{1}\s\w{1}",
]

def is_garbage(text: str) -> bool:
    if not text or len(text.strip()) < 10:
        return True
    for pattern in GARBAGE_PATTERNS:
        if re.search(pattern, text):
            return True
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / len(text) > 0.20:
        return True
    return False


langchain_docs = []
skipped = []

for record in data:
    text = record["text"] or ""

    # skip empty
    if not text.strip():
        skipped.append((record["type"], "empty text"))
        continue

    # skip garbage images
    if record["type"] == "Image" and is_garbage(text):
        skipped.append(("Image", f"page={record['metadata'].get('page_number')} garbage OCR"))
        continue

    langchain_docs.append(Document(
        page_content=text,
        metadata={
            "type"       : record["type"],
            "page"       : record["metadata"].get("page_number"),
            "element_id" : record["element_id"],
            "image_path" : record["metadata"].get("image_path"),
        }
    ))

# what was skipped
print(f"Skipped ({len(skipped)}):")
for t, reason in skipped:
    print(f"  [{t}] {reason}")

# what went in
type_counts = Counter(d.metadata["type"] for d in langchain_docs)
print(f"\nTotal docs in FAISS: {len(langchain_docs)}")
for t, c in sorted(type_counts.items()):
    print(f"  {t:20s} : {c}")

vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"\nFAISS index size: {vectorstore.index.ntotal}")


Skipped (20):
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [Image] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [Image] page=13 garbage OCR
  [PageBreak] empty text
  [Image] page=14 garbage OCR
  [PageBreak] empty text
  [Image] page=15 garbage OCR
  [Image] page=15 garbage OCR
  [PageBreak] empty text

Total docs in FAISS: 227
  FigureCaption        : 5
  Footer               : 7
  Formula              : 4
  Header               : 3
  Image                : 2
  ListItem             : 43
  NarrativeText        : 81
  Table                : 4
  Title                : 26
  UncategorizedText    : 52

FAISS index size: 227


In [36]:
from pathlib import Path

# define path
vectordb_path = (Path.cwd() / ".." / "data" / "vectordb").resolve()
vectordb_path.mkdir(parents=True, exist_ok=True)

# build vectorstore
vectorstore = FAISS.from_documents(langchain_docs, embeddings)
print(f"FAISS index size: {vectorstore.index.ntotal}")

# save to disk
vectorstore.save_local(str(vectordb_path))
print(f"Vectorstore saved to: {vectordb_path}")

FAISS index size: 227
Vectorstore saved to: D:\PROJECTS\multimodal_rag\data\vectordb


In [37]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

vectordb_path = (Path.cwd() / ".." / "data" / "vectordb").resolve()

vectorstore = FAISS.load_local(
    str(vectordb_path),
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vectorstore | index size: {vectorstore.index.ntotal}")


Loaded vectorstore | index size: 227


In [38]:
import base64
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

# embeddings + vectorstore
embeddings   = NVIDIAEmbeddings(model="baai/bge-m3")
vectordb_path = (Path.cwd() / ".." / "data" / "vectordb").resolve()

vectorstore = FAISS.load_local(
    str(vectordb_path),
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Vectorstore loaded | index size: {vectorstore.index.ntotal}")

# text LLM for RAG
text_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

# vision LLM for images
vision_llm = ChatNVIDIA(model="nvidia/llama-3.2-90b-vision-instruct")

Vectorstore loaded | index size: 227


d:\PROJECTS\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Model nvidia/llama-3.2-90b-vision-instruct is unknown, check `available_models`. Inference may fail.
  warnings.warn(


In [39]:
def retrieve_with_images(query: str, k: int = 4) -> tuple[list, list]:
    results = vectorstore.similarity_search_with_score(query, k=k)

    text_docs  = []
    image_docs = []

    for doc, score in results:
        doc.metadata["score"] = round(score, 4)
        if doc.metadata.get("type") == "Image":
            image_docs.append(doc)
        else:
            text_docs.append(doc)

    return text_docs, image_docs


def print_retrieved(text_docs: list, image_docs: list) -> None:
    print(f"\nText docs retrieved  : {len(text_docs)}")
    for i, doc in enumerate(text_docs, start=1):
        print(f"\n  [{i}] type={doc.metadata['type']} | page={doc.metadata['page']} | score={doc.metadata['score']}")
        print(f"       {doc.page_content[:200]}")

    print(f"\nImage docs retrieved : {len(image_docs)}")
    for i, doc in enumerate(image_docs, start=1):
        print(f"\n  [{i}] page={doc.metadata['page']} | score={doc.metadata['score']}")
        print(f"       path={doc.metadata.get('image_path')}")
        print(f"       ocr ={doc.page_content[:100]}")

In [40]:
def answer_from_text(query: str, text_docs: list) -> str:
    context = "\n\n".join(
        f"[Page {d.metadata['page']} | {d.metadata['type']}]\n{d.page_content}"
        for d in text_docs
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", 
         "Answer the question based only on the context below.\n"
         "If the answer is not in the context, say 'Not found in document'.\n\n"
         "Context:\n{context}"),
        ("human", "{question}"),
    ])

    chain = prompt | text_llm | StrOutputParser()
    return chain.invoke({"context": context, "question": query})

In [41]:
def answer_from_images(query: str, image_docs: list) -> str:
    if not image_docs:
        return "No relevant images found."

    answers = []
    for doc in image_docs:
        image_path = doc.metadata.get("image_path")
        if not image_path or not Path(image_path).exists():
            continue

        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        ext      = Path(image_path).suffix.lower().replace(".", "")
        mime     = f"image/{'jpeg' if ext in ['jpg','jpeg'] else ext}"

        response = vision_llm.invoke([
            HumanMessage(content=[
                {"type": "text",      "text": f"This image is from page {doc.metadata['page']} of a research paper. {query}"},
                {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
            ])
        ])
        answers.append(f"[Image page {doc.metadata['page']}]: {response.content}")

    return "\n\n".join(answers) if answers else "No valid image paths found."

In [42]:
def multimodal_chat(query: str, k: int = 4) -> None:
    print(f"QUERY: {query}")

    text_docs, image_docs = retrieve_with_images(query, k=k)
    print_retrieved(text_docs, image_docs)

    # text answer
    text_answer = answer_from_text(query, text_docs)
    print(text_answer)

    # vision answer if images were retrieved
    if image_docs:
        print(f"\n--- VISION ANSWER (from retrieved images) ---")
        vision_answer = answer_from_images(query, image_docs)
        print(vision_answer)

In [ ]:
multimodal_chat( "What is the transformer architecture?", k=4)


QUERY: What is the transformer architecture?

Text docs retrieved  : 4

  [1] type=FigureCaption | page=3 | score=0.8349999785423279
       Figure 1: The Transformer - model architecture.

  [2] type=NarrativeText | page=2 | score=0.867900013923645
       In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Tran

  [3] type=NarrativeText | page=3 | score=0.8992000222206116
       The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, 

  [4] type=NarrativeText | page=1 | score=0.9236000180244446
       The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and

Im

In [45]:
# questions = [
#     "What is the transformer architecture?",
#     "explain the positional encoding",
#     "What is Multi-Head Attention",
#     "What does the model architecture diagram show?",
#     "What are the results on WMT translation tasks?",
# ]

# for q in questions:
#     multimodal_chat(q, k=4)

In [46]:
multimodal_chat( "explain the positional encoding", k=4)

QUERY: explain the positional encoding

Text docs retrieved  : 4

  [1] type=Title | page=6 | score=0.5831999778747559
       3.5 Positional Encoding

  [2] type=NarrativeText | page=6 | score=0.6693000197410583
       where pos is the position and i is the dimension. That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We c

  [3] type=NarrativeText | page=6 | score=0.6855000257492065
       Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of t

  [4] type=NarrativeText | page=6 | score=0.8791999816894531
       We also experimented with using learned positional embeddings [9] instead, and found that the two versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version beca

Image docs retrieved : 0
Positional enc

In [47]:
multimodal_chat( "What is Multi-Head Attention", k=4)

QUERY: What is Multi-Head Attention

Text docs retrieved  : 4

  [1] type=UncategorizedText | page=4 | score=0.35659998655319214
       Multi-Head Attention

  [2] type=Title | page=4 | score=0.6097000241279602
       3.2.2 Multi-Head Attention

  [3] type=NarrativeText | page=5 | score=0.6474000215530396
       Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this.

  [4] type=NarrativeText | page=4 | score=0.6769999861717224
       Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel.

Image docs retrieved : 0
Multi-Head Attention is a mechanism that allows a model to jointly attend to information from different representation subspaces at different positions.


In [48]:
multimodal_chat( "What does the model architecture diagram show?", k=4)

QUERY: What does the model architecture diagram show?

Text docs retrieved  : 4

  [1] type=Title | page=2 | score=0.714900016784668
       3 Model Architecture

  [2] type=FigureCaption | page=3 | score=0.7300999760627747
       Figure 1: The Transformer - model architecture.

  [3] type=NarrativeText | page=7 | score=0.9168999791145325
       This section describes the training regime for our models.

  [4] type=NarrativeText | page=2 | score=0.9564999938011169
       Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of conti

Image docs retrieved : 0
The model architecture diagram, Figure 1, shows The Transformer model architecture.


In [49]:
multimodal_chat( "What are the results on WMT translation tasks?", k=4)

QUERY: What are the results on WMT translation tasks?

Text docs retrieved  : 4

  [1] type=NarrativeText | page=10 | score=0.7797999978065491
       For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-Fre

  [2] type=NarrativeText | page=8 | score=0.9182000160217285
       On the WMT 2014 English-to-French translation task, our big model achieves a BLEU score of 41.0, outperforming all of the previously published single models, at less than 1/4 the training cost of the 

  [3] type=NarrativeText | page=7 | score=0.9470000267028809
       We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million sentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source- target vocabu

  [4] type=Title | page=8 | score=0.958299994468689
       6.1 Machine Translation

Image docs retrieved : 0

In [53]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# list all available models and filter vision ones
all_models = ChatNVIDIA.get_available_models()

vision_models = [
    m for m in all_models
    if any(k in m.id.lower() for k in ["neva", "vision", "vlm", "visual", "llava", "llama-3.2"])
]

print("Available vision models:")
for m in vision_models:
    print(f"  {m.id}")

Available vision models:
  meta/llama-3.2-11b-vision-instruct
  meta/llama-3.2-90b-vision-instruct
  microsoft/phi-3-vision-128k-instruct
  meta/llama-3.2-1b-instruct
  meta/llama-3.2-3b-instruct
  microsoft/phi-3.5-vision-instruct


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
import base64
from pathlib import Path

vision_llm = ChatNVIDIA(model="nvidia/neva-22b")

def answer_from_images(query: str, image_docs: list) -> str:
    if not image_docs:
        return "No relevant images found."

    answers = []
    for doc in image_docs:
        image_path = doc.metadata.get("image_path")
        if not image_path or not Path(image_path).exists():
            continue

        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        ext  = Path(image_path).suffix.lower().replace(".", "")
        mime = f"image/{'jpeg' if ext in ['jpg', 'jpeg'] else ext}"

        response = vision_llm.invoke([
            HumanMessage(content=[
                {"type": "text",      "text": f"This is from page {doc.metadata['page']} of a research paper. {query}"},
                {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
            ])
        ])
        answers.append(f"[Image page {doc.metadata['page']}]: {response.content}")

    return "\n\n".join(answers) if answers else "No valid image paths found."

In [50]:
import base64
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# --- build a separate image-only vectorstore ---
image_docs_only = [
    doc for doc in langchain_docs
    if doc.metadata.get("type") == "Image"
]

print(f"Image docs in separate store: {len(image_docs_only)}")
for doc in image_docs_only:
    print(f"  page={doc.metadata['page']} | {doc.page_content[:80]}")

# separate FAISS just for images
image_vectorstore = FAISS.from_documents(image_docs_only, embeddings)

# main retriever — text only
text_retriever  = vectorstore.as_retriever(search_kwargs={"k": 4})

# image retriever — always pulls all available images
image_retriever = image_vectorstore.as_retriever(search_kwargs={"k": len(image_docs_only)})

Image docs in separate store: 2
  page=3 | Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, 
  page=4 | EEE Scaled Dot-Product | Attention 4


In [51]:
def retrieve_with_images(query: str) -> tuple[list, list]:
    text_docs  = text_retriever.invoke(query)
    image_docs = image_retriever.invoke(query)  # always returns all images

    print(f"\nText docs  : {len(text_docs)}")
    print(f"Image docs : {len(image_docs)}")

    for i, doc in enumerate(text_docs, start=1):
        print(f"  text [{i}] page={doc.metadata['page']} | {doc.page_content[:120]}")

    for i, doc in enumerate(image_docs, start=1):
        print(f"  img  [{i}] page={doc.metadata['page']} | path={doc.metadata.get('image_path')}")

    return text_docs, image_docs

In [54]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# list all available models and filter vision ones
all_models = ChatNVIDIA.get_available_models()

vision_models = [
    m for m in all_models
    if any(k in m.id.lower() for k in ["neva", "vision", "vlm", "visual", "llava", "llama-3.2"])
]

print("Available vision models:")
for m in vision_models:
    print(f"  {m.id}")

Available vision models:
  meta/llama-3.2-11b-vision-instruct
  meta/llama-3.2-90b-vision-instruct
  microsoft/phi-3-vision-128k-instruct
  meta/llama-3.2-1b-instruct
  meta/llama-3.2-3b-instruct
  microsoft/phi-3.5-vision-instruct
